# Generating Embeddings with Hugging Face (Free Models)

**Runs on:** Google Colab (free CPU/GPU), no API key needed.

### What you will learn
1. What an embedding is
2. Generating embeddings with `sentence-transformers`
3. Comparing sentences with cosine similarity
4. Visualising embeddings in 2D
5. Building a mini semantic search engine and an FAQ bot
6. Comparing two embedding models

## 1. What is an Embedding?
An embedding is a list of numbers (a vector) that represents the **meaning** of a text.
Similar meaning gives close vectors, different meaning gives distant vectors.

In [ ]:
!pip install -q sentence-transformers scikit-learn matplotlib pandas

## 2. Load a Free Model
`all-MiniLM-L6-v2` is small (~80 MB), fast, and outputs **384-dimensional** vectors.

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
print("Model loaded. Embedding dimension:", model.get_sentence_embedding_dimension())

## 3. Your First Embedding

In [ ]:
sentence = "Cricket is the most popular sport in India."
embedding = model.encode(sentence)

print("Sentence:", sentence)
print("Shape:", embedding.shape)
print("First 10 values:", embedding[:10].round(4))

## 4. Embeddings for Many Sentences

In [ ]:
sentences = [
    "The chef is cooking pasta in the kitchen",
    "A cook is preparing noodles",
    "The rocket was launched into space",
    "NASA sent a satellite into orbit",
    "My laptop battery drains very fast",
]

embeddings = model.encode(sentences)
print("Matrix shape (sentences x dimensions):", embeddings.shape)

## 5. Similarity Between Sentences

In [ ]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

sim = cosine_similarity(embeddings)
pd.DataFrame(sim, index=sentences, columns=range(1, len(sentences) + 1)).round(2)

The cooking pair (1, 2) and the space pair (3, 4) should score noticeably higher with each other than with the laptop sentence.

## 6. Visualise in 2D with PCA

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

points = PCA(n_components=2).fit_transform(embeddings)

plt.figure(figsize=(8, 5))
plt.scatter(points[:, 0], points[:, 1])
for i, s in enumerate(sentences):
    plt.annotate(s, (points[i, 0], points[i, 1]), fontsize=8)
plt.title("Sentence embeddings in 2D (PCA)")
plt.grid(True)
plt.show()

## 7. Mini Semantic Search Engine

In [ ]:
documents = [
    "Python is a popular programming language for data science.",
    "The monsoon season brings heavy rainfall to South India.",
    "Neural networks are trained using backpropagation.",
    "Virat Kohli scored a century in the final match.",
    "The Eiffel Tower attracts millions of tourists each year.",
    "Photosynthesis lets plants convert sunlight into energy.",
]
doc_embeddings = model.encode(documents)

def semantic_search(query, top_k=2):
    q = model.encode([query])
    scores = cosine_similarity(q, doc_embeddings)[0]
    best = scores.argsort()[::-1][:top_k]
    print(f"Query: {query}")
    for i in best:
        print(f"  {scores[i]:.3f} | {documents[i]}")
    print()

semantic_search("How do deep learning models learn?")
semantic_search("Which cricketer made a hundred?")
semantic_search("How do plants make food?")

## 8. Exercise: Simple FAQ Bot
Store question and answer pairs, embed the **questions**, and return the answer of the closest question.

In [ ]:
faq = {
    "How can I reset my password?":        "Click 'Forgot password' on the login page and follow the email link.",
    "What are your working hours?":        "We are open Monday to Friday, 9 AM to 6 PM.",
    "How do I track my order?":            "Go to 'My Orders' and click 'Track' next to your order.",
    "Can I return a product?":             "Yes, returns are accepted within 30 days of delivery.",
    "Which payment methods do you accept?": "We accept cards, UPI, and net banking.",
}
faq_questions = list(faq.keys())
faq_embeddings = model.encode(faq_questions)

def faq_bot(user_question):
    scores = cosine_similarity(model.encode([user_question]), faq_embeddings)[0]
    best = scores.argmax()
    print(f"User   : {user_question}")
    print(f"Matched: {faq_questions[best]} (score {scores[best]:.2f})")
    print(f"Bot    : {faq[faq_questions[best]]}\n")

faq_bot("I forgot my login password")
faq_bot("where is my parcel right now?")
faq_bot("do you take UPI?")

## 9. Exercise: Compare Two Models

In [ ]:
import time

queries = ["How do plants make food?", "Which cricketer made a hundred?"]

for name in ["all-MiniLM-L6-v2", "all-mpnet-base-v2"]:
    m = SentenceTransformer(name)
    start = time.time()
    d = m.encode(documents)
    print(f"Model: {name} | dimension: {d.shape[1]} | encode time: {time.time()-start:.2f}s")
    for q in queries:
        s = cosine_similarity(m.encode([q]), d)[0]
        top = s.argmax()
        print(f"   '{q}' -> {s[top]:.3f} | {documents[top]}")
    print()

## Key Takeaways

| Concept | Meaning |
|---|---|
| Embedding | Numeric vector representing the meaning of text |
| `all-MiniLM-L6-v2` | Free, small, fast, 384 dimensions |
| `all-mpnet-base-v2` | Free, larger, usually more accurate, 768 dimensions |
| Cosine similarity | Measures closeness of two embeddings |
| Semantic search | Retrieves by meaning, not exact words |
| PCA | Only for visualising high-dimensional vectors in 2D |